In [ ]:
import cv2
import numpy as np
import openvino as ov
import time

# 1. Initialize Core and Load Model
core = ov.Core()
model = core.read_model("yolo26m_openvino_model/yolo26m.xml")

# 2. Fix Dynamic Shape & Preprocessing
# Reshape to a static 640x640 for optimization
model.reshape({model.input().get_any_name(): [1, 3, 640, 640]})

ppp = ov.preprocess.PrePostProcessor(model)
ppp.input().tensor().set_element_type(ov.Type.u8).set_layout(ov.Layout("NHWC")).set_color_format(ov.preprocess.ColorFormat.BGR)
ppp.input().model().set_layout(ov.Layout("NCHW"))
model = ppp.build()

compiled_model = core.compile_model(model, "AUTO")
infer_request = compiled_model.create_infer_request()

# 3. Letterbox Helper
def letterbox(img, new_shape=(640, 640), color=(114, 114, 114)):
    shape = img.shape[:2]
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]
    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    return cv2.copyMakeBorder(img, 0, dh, 0, dw, cv2.BORDER_CONSTANT, value=color), r

# 4. Inference Loop
cap = cv2.VideoCapture("video/hongkong2.mp4")

while cap.isOpened():
    120
    ret, frame = cap.read()
    if not ret: break

    # Pre-process
    input_tensor, ratio = letterbox(frame)
    input_tensor = input_tensor[np.newaxis, ...] # Add batch dimension [1, 640, 640, 3]

    # Inference
    results = infer_request.infer({0: input_tensor})
    output = results[compiled_model.output(0)] # Shape: [1, 84, 8400]

    # Post-process (Parsing YOLOv26)
    predictions = np.squeeze(output).T # Shape: [8400, 84]
    
    for pred in predictions:
        scores = pred[4:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        if class_id == 9 and confidence > 0.5:
            x, y, w, h = pred[:4]
            # # Rescale to original frame
            # left = int((x - w / 2) / ratio)
            # top = int((y - h / 2) / ratio)
            # width, height = int(w / ratio), int(h / ratio)

            # cv2.rectangle(frame, (left, top), (left + width, top + height), (0, 255, 0), 2)

    # cv2.imshow("OpenVINO YOLOv26m Python", frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'): break
    end_time = time.perf_counter()
    print(f"Execution time: {end_time - start_time:.4f} seconds")

cap.release()
cv2.destroyAllWindows()

Execution time: 0.1214 seconds
Execution time: 0.0818 seconds
Execution time: 0.0842 seconds
Execution time: 0.0858 seconds
Execution time: 0.0836 seconds
Execution time: 0.0853 seconds
Execution time: 0.0850 seconds
Execution time: 0.0831 seconds
Execution time: 0.0851 seconds
Execution time: 0.0861 seconds
Execution time: 0.0839 seconds
Execution time: 0.0814 seconds
Execution time: 0.0822 seconds
Execution time: 0.0836 seconds
Execution time: 0.0843 seconds
Execution time: 0.0866 seconds
Execution time: 0.0845 seconds
Execution time: 0.0854 seconds
Execution time: 0.0851 seconds
Execution time: 0.0835 seconds
Execution time: 0.0840 seconds
Execution time: 0.0842 seconds
Execution time: 0.0826 seconds
Execution time: 0.0823 seconds
Execution time: 0.0833 seconds
Execution time: 0.0835 seconds
Execution time: 0.0840 seconds
Execution time: 0.0858 seconds
Execution time: 0.0822 seconds
Execution time: 0.0860 seconds
Execution time: 0.0833 seconds
Execution time: 0.0822 seconds
Executio

KeyboardInterrupt: 

In [ ]:
import openvino as ov

# 1. Compile the model (using AUTO allows OpenVINO to pick the best device)
compiled_model = core.compile_model(model, "AUTO")

# 2. Query the actual execution device
execution_device = compiled_model.get_property("EXECUTION_DEVICE")

# 3. Log the result
if "GPU" in execution_device:
    logger.info(f"✅ SUCCESS: Inference is running on the Intel GPU ({execution_device})")
else:
    logger.warning(f"⚠️ ATTENTION: Inference is running on {execution_device}. GPU was not used.")

# 4. Optional: Check detailed hardware name
full_device_name = core.get_property(execution_device, "FULL_DEVICE_NAME")
logger.info(f"Device Details: {full_device_name}")

RuntimeError: Exception from src/inference/src/cpp/compiled_model.cpp:140:
Exception from src/plugins/auto/src/auto_compiled_model.cpp:259:
AUTO: not supported property EXECUTION_DEVICE



In [ ]:
import cv2
import numpy as np
import openvino as ov
import logging
import sys

# ==========================================
# 1. SETUP LOGGING (IPYNB FRIENDLY)
# ==========================================
logger = logging.getLogger("TrafficLightLogger")
if logger.hasHandlers(): logger.handlers.clear()
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
logger.addHandler(handler)

# ==========================================
# 2. INITIALIZE OPENVINO & HARDWARE
# ==========================================
core = ov.Core()
model_path = "yolo26m_openvino_model/yolo26m.xml"
video_path = "video/hongkong2.mp4"

model = core.read_model(model_path)

# Fix Dynamic Shape to Static 640x640 for GPU optimization
model.reshape({model.input().get_any_name(): [1, 3, 640, 640]})

# PrePostProcessor: Handles BGR->RGB and NHWC->NCHW on the hardware
ppp = ov.preprocess.PrePostProcessor(model)
ppp.input().tensor().set_element_type(ov.Type.u8).set_layout(ov.Layout("NHWC")).set_color_format(ov.preprocess.ColorFormat.BGR)
ppp.input().model().set_layout(ov.Layout("NCHW"))
model = ppp.build()

# Compile for GPU (Falls back to CPU if GPU fails)
device_to_use = "GPU" if "GPU" in core.available_devices else "CPU"
compiled_model = core.compile_model(model, device_to_use)

# Verify which device is actually used
execution_device = compiled_model.get_property("EXECUTION_DEVICES")
logger.info(f"✅ Model compiled for: {execution_device}")

# ==========================================
# 3. HELPERS
# ==========================================
def letterbox(img, new_shape=(640, 640)):
    shape = img.shape[:2]
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    canvas = np.full((new_shape[0], new_shape[1], 3), 114, dtype=np.uint8)
    canvas[:new_unpad[1], :new_unpad[0]] = img
    return canvas, r

TRAFFIC_LIGHT_ID = 9
infer_request = compiled_model.create_infer_request()

# ==========================================
# 4. MAIN INFERENCE LOOP
# ==========================================
cap = cv2.VideoCapture(video_path)
count_history = -1

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Pre-process
    input_tensor, ratio = letterbox(frame)
    input_tensor = input_tensor[np.newaxis, ...]

    # Infer
    results = infer_request.infer({0: input_tensor})
    output = results[compiled_model.output(0)]
    
    # YOLOv26 Output Parsing [1, 84, 8400]
    predictions = np.squeeze(output).T
    
    boxes, confs = [], []
    for pred in predictions:
        scores = pred[4:]
        class_id = np.argmax(scores)
        conf = scores[class_id]
        
        # FILTER: Only Traffic Lights
        if class_id == TRAFFIC_LIGHT_ID and conf > 0.5:
            x, y, w, h = pred[:4]
            left = int((x - w / 2) / ratio)
            top = int((y - h / 2) / ratio)
            boxes.append([left, top, int(w / ratio), int(h / ratio)])
            confs.append(float(conf))

    # NMS (Non-Maximum Suppression) to remove overlaps
    indices = cv2.dnn.NMSBoxes(boxes, confs, 0.5, 0.45)
    
    # Logging Logic
    current_count = len(indices)
    if current_count != count_history:
        logger.info(f"Traffic Lights in frame: {current_count}")
        count_history = current_count

    # Visualization
    for i in indices:
        box = boxes[i]
        cv2.rectangle(frame, (box[0], box[1]), (box[0]+box[2], box[1]+box[3]), (0, 0, 255), 2)
        cv2.putText(frame, f"Light {confs[i]:.2f}", (box[0], box[1]-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    cv2.imshow("Intel GPU Traffic Light Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
logger.info("Inference complete.")

2026-03-10 15:17:18,213 | INFO | ✅ Model compiled for: ['GPU.0']
2026-03-10 15:17:18,430 | INFO | Traffic Lights in frame: 2
2026-03-10 15:17:18,511 | INFO | Traffic Lights in frame: 0
2026-03-10 15:17:23,700 | INFO | Traffic Lights in frame: 1
2026-03-10 15:17:23,774 | INFO | Traffic Lights in frame: 0
2026-03-10 15:17:34,207 | INFO | Traffic Lights in frame: 1
2026-03-10 15:17:34,281 | INFO | Traffic Lights in frame: 0
2026-03-10 15:17:39,459 | INFO | Traffic Lights in frame: 1
2026-03-10 15:17:39,535 | INFO | Traffic Lights in frame: 0
2026-03-10 15:17:39,616 | INFO | Inference complete.
